In [3]:
from dotenv import load_dotenv
load_dotenv()
import os
from groq import Groq
from typing import List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from datetime import datetime

# Configuración
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Modelos base
class Research(BaseModel):
    content: str = Field(description="Research content about the topic")
    sources: List[str] = Field(description="Sources used in research", default_factory=list)
    timestamp: datetime = Field(default_factory=datetime.now)

class Article(BaseModel):
    content: str = Field(description="Article content")
    version: int = Field(description="Version number of the article")
    feedback: str = Field(description="Editor feedback for this version", default="")
    
    @property
    def formatted(self) -> str:
        return f"Version: {self.version}\nContent: {self.content}\nFeedback: {self.feedback}"

class ArticleState(TypedDict):
    topic: str
    research_data: Research | None
    articles: List[Article]
    editor_feedback: str
    max_versions: int

# Prompts
research_prompt = """Conduct thorough research on the following topic:
{topic}

Please provide:
1. Comprehensive background
2. Current state
3. Key insights
4. Relevant data
5. Important sources

Structure your research clearly and professionally."""

writing_prompt = """Write version {version} of an article about: {topic}

Based on this research:
{research}

Previous feedback to address:
{feedback}

Requirements:
1. Clear and engaging writing
2. Professional tone
3. Well-structured content
4. Address previous feedback
5. Show improvement over previous versions"""

def do_research(state: ArticleState):
    """Research node"""
    print("Starting research...")  # Debug
    topic = state['topic']
    
    completion = llm.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[{"role": "system", "content": research_prompt.format(topic=topic)}],
        temperature=0.7
    )
    
    research = Research(
        content=completion.choices[0].message.content
    )
    
    new_state = state.copy()
    new_state['research_data'] = research
    print("Research completed")  # Debug
    return new_state

def write_article(state: ArticleState):
    """Writing node"""
    print("Starting article writing...")  # Debug
    print(f"Current state keys: {state.keys()}")  # Debug
    
    if not state.get('research_data'):
        raise ValueError("No research data available")
        
    version = len(state['articles']) + 1
    feedback = state.get('editor_feedback', '')
    
    completion = llm.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[{
            "role": "system", 
            "content": writing_prompt.format(
                version=version,
                topic=state['topic'],
                research=state['research_data'].content,
                feedback=feedback
            )
        }],
        temperature=0.7
    )
    
    article = Article(
        content=completion.choices[0].message.content,
        version=version,
        feedback=""
    )
    
    new_state = state.copy()
    articles = new_state['articles'].copy()
    articles.append(article)
    new_state['articles'] = articles
    
    print(f"Article version {version} completed")  # Debug
    return new_state

def human_review(state: ArticleState):
    """Human review node - interruption point"""
    return state

def should_continue(state: ArticleState):
    """Determine next step based on feedback and version count"""
    if not state.get('editor_feedback'):
        return END
        
    if len(state['articles']) >= state['max_versions']:
        print("\nMaximum versions reached!")
        return END
        
    return "write_article"

# Construcción del grafo
builder = StateGraph(ArticleState)

# Agregar nodos
builder.add_node("do_research", do_research)
builder.add_node("write_article", write_article)
builder.add_node("human_review", human_review)

# Configurar flujo
builder.add_edge(START, "do_research")
builder.add_edge("do_research", "write_article")
builder.add_edge("write_article", "human_review")
builder.add_conditional_edges(
    "human_review",
    should_continue,
    {
        "write_article": "write_article",
        "END": END
    }
)

# Compilar grafo
graph = builder.compile(
    interrupt_before=['human_review'],
    checkpointer=MemorySaver()
)
def main():
    # Configuración inicial
    topic = input("Enter topic: ")
    max_versions = 5
    thread = {"configurable": {"thread_id": "1"}}
    
    # Estado inicial
    initial_state: ArticleState = {
        "topic": topic,
        "research_data": None,
        "articles": [],
        "editor_feedback": "",
        "max_versions": max_versions
    }
    
    try:
        # Primera ejecución
        print("\nStarting research and writing first version...")
        for event in graph.stream(initial_state, thread, stream_mode="values"):
            # Verificamos si hay artículos en el evento
            if 'articles' in event and event['articles']:
                article = event['articles'][-1]
                print(f"\n=== Article Version {article.version} ===")
                print(article.formatted)
                print("\n" + "="*50)
    
        while True:
            print("\nOptions:")
            print("1. Accept")
            print("2. Reject")
            print("3. Request revision")
            
            choice = input("\nChoice (1-3): ")
            
            if choice in ["1", "2"]:
                # Mostrar versión final
                if choice == "1":
                    print("\nVersion accepted")
                    
                else:
                    print("\nVersion rejected")
                break
                    
            feedback = input("Provide feedback for revision: ")
            print(f'Feedback: {feedback}')
            # Actualizar estado con feedback
            graph.update_state(
                thread, 
                {"editor_feedback": feedback}, 
                as_node="human_review"
            )
            
            print("\nGenerating new version based on feedback...")
            # Continuar ejecución
            for event in graph.stream(None, thread, stream_mode="values"):
                if 'articles' in event and event['articles']:
                    article = event['articles'][-1]
                    print(f"\n=== Article Version {article.version} ===")
                    print(article.formatted)
                    print("\n" + "="*50)
    
    except Exception as e:
        print(f"\nError during execution: {str(e)}")
        # Intentar mostrar la última versión incluso si hay error
        try:
            current_state = graph.get_state(thread)
            if current_state and current_state.values.get('articles'):
                print("\nLast saved version:")
                print("=" * 50)
                print(current_state.values['articles'][-1].formatted)
                print("=" * 50)
        except:
            print("Could not retrieve last version")
        
        print("\nDebug information:")
        print("Current state:", graph.get_state(thread).values if thread else "No state available")

if __name__ == "__main__":
    main()


Starting research and writing first version...
Starting research...
Research completed
Starting article writing...
Current state keys: dict_keys(['topic', 'research_data', 'articles', 'editor_feedback', 'max_versions'])
Article version 1 completed

=== Article Version 1 ===
Version: 1
Content: Title: Claude: A Name of French Origin with a Rich History

Introduction

Claude is a unisex given name of French origin, meaning "cloth" or "steward." This name has been popular in France and other French-speaking countries, as well as in English-speaking countries. Over the years, the name Claude has been given to various notable individuals in different fields, including art, literature, music, and politics.

Notable Individuals Named Claude

Claude Monet (1840-1926) was a renowned French impressionist painter, whose works significantly contributed to the development of the impressionist style of painting. Monet is best known for his series of paintings depicting water lilies, haystacks, and 